<a href="https://colab.research.google.com/github/jtista/BellaBeat-/blob/main/BellaBeat_SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sqlite3
import pandas as pd

In [ ]:
# Creating a local SQLite database
conn = sqlite3.connect('bellabeat.db')

In [ ]:
activity_sleep = pd.read_csv('/content/drive/MyDrive/Bellabeat/cleaned_sleepDay_merged.csv')
weight_log     = pd.read_csv('/content/drive/MyDrive/Bellabeat/cleaned_weight_log.csv')


In [ ]:
# Push them into SQLite as tables
activity_sleep.to_sql('sleepDay', conn, if_exists='replace', index=False)
weight_log.to_sql('weight_log',         conn, if_exists='replace', index=False)

67

In [ ]:
# verify tables loaded correctly
def run_query(query):
    return pd.read_sql_query(query, conn)

In [ ]:
run_query("SELECT COUNT(*) AS total_rows FROM sleepDay")

,total_rows
0,863


In [ ]:
run_query("SELECT COUNT(*) AS total_rows FROM weight_log")

,total_rows
0,67


In [ ]:
# Preview first few rows
run_query("SELECT * FROM sleepDay LIMIT 5")

,id,activitydate,totalsteps,totaldistance,trackerdistance,loggedactivitiesdistance,veryactivedistance,moderatelyactivedistance,lightactivedistance,sedentaryactivedistance,...,fairlyactiveminutes,lightlyactiveminutes,sedentaryminutes,calories,day_of_week,month,total_minutes_asleep,total_minutes_in_bed,total_hours_asleep,total_hours_in_bed
0,1503960366,12-04-2016,13162,8.50,8.50,0.0,1.88,0.55,6.06,0.0,...,13,328,728,1985,Tuesday,April,327.0,346.0,5.4,5.8
1,1503960366,13-04-2016,10735,6.97,6.97,0.0,1.57,0.69,4.71,0.0,...,19,217,776,1797,Wednesday,April,384.0,407.0,6.4,6.8
2,1503960366,14-04-2016,10460,6.74,6.74,0.0,2.44,0.40,3.91,0.0,...,11,181,1218,1776,Thursday,April,NaN,NaN,NaN,NaN
3,1503960366,15-04-2016,9762,6.28,6.28,0.0,2.14,1.26,2.83,0.0,...,34,209,726,1745,Friday,April,412.0,442.0,6.9,7.4
4,1503960366,16-04-2016,12669,8.16,8.16,0.0,2.71,0.41,5.04,0.0,...,10,221,773,1863,Saturday,April,340.0,367.0,5.7,6.1


In [ ]:
run_query("""
    SELECT
        id,
        ROUND(AVG(totalsteps), 0)              AS avg_steps,
        ROUND(AVG(calories), 0)                AS avg_calories,
        ROUND(AVG(total_minutes_asleep), 0)    AS avg_minutes_asleep
    FROM sleepDay
    GROUP BY id
    ORDER BY avg_steps DESC
""")

,id,avg_steps,avg_calories,avg_minutes_asleep
0,8877689391,16040.0,3420.0,NaN
1,8053475328,14763.0,2946.0,297.0
2,1503960366,12521.0,1877.0,360.0
3,7007744171,12267.0,2686.0,69.0
4,2022484408,11371.0,2510.0,NaN
5,3977333714,10985.0,1514.0,294.0
6,4388161847,10814.0,3094.0,400.0
7,6962181067,9795.0,1982.0,448.0
8,7086361926,9684.0,2598.0,453.0
9,2347167796,9520.0,2043.0,447.0


In [ ]:
run_query("""
    SELECT
        id,
        ROUND(AVG(totalsteps), 0) AS avg_steps,
        CASE
            WHEN AVG(totalsteps) < 5000  THEN 'Sedentary'
            WHEN AVG(totalsteps) < 7500  THEN 'Low Active'
            WHEN AVG(totalsteps) < 10000 THEN 'Fairly Active'
            ELSE 'Very Active'
        END AS activity_level
    FROM sleepDay
    GROUP BY id
    ORDER BY avg_steps DESC
""")

,id,avg_steps,activity_level
0,8877689391,16040.0,Very Active
1,8053475328,14763.0,Very Active
2,1503960366,12521.0,Very Active
3,7007744171,12267.0,Very Active
4,2022484408,11371.0,Very Active
5,3977333714,10985.0,Very Active
6,4388161847,10814.0,Very Active
7,6962181067,9795.0,Fairly Active
8,7086361926,9684.0,Fairly Active
9,2347167796,9520.0,Fairly Active


In [ ]:
run_query("""
    SELECT
        day_of_week,
        ROUND(AVG(totalsteps), 0)   AS avg_steps,
        ROUND(AVG(calories), 0)     AS avg_calories,
        COUNT(*)                    AS total_entries
    FROM sleepDay
    GROUP BY day_of_week
    ORDER BY avg_steps DESC
""")

,day_of_week,avg_steps,avg_calories,total_entries
0,Tuesday,8949.0,2441.0,138
1,Saturday,8947.0,2429.0,113
2,Monday,8488.0,2386.0,110
3,Thursday,8185.0,2274.0,133
4,Wednesday,8158.0,2339.0,139
5,Friday,7821.0,2352.0,120
6,Sunday,7627.0,2311.0,110


### Sleep Analysis

In [ ]:
run_query("""
    SELECT
        ROUND(AVG(total_minutes_asleep) / 60.0, 2)  AS avg_hours_asleep,
        ROUND(AVG(total_minutes_in_bed) / 60.0, 2)  AS avg_hours_in_bed,
        COUNT(CASE WHEN total_minutes_asleep >= 420
              THEN 1 END)                            AS days_with_7hrs_sleep,
        COUNT(total_minutes_asleep)                  AS total_sleep_records
    FROM sleepDay
    WHERE total_minutes_asleep IS NOT NULL
""")

,avg_hours_asleep,avg_hours_in_bed,days_with_7hrs_sleep,total_sleep_records
0,6.99,7.64,229,410


### STEPS VS CALORIES CORRELATION

In [ ]:
run_query("""
    SELECT
        totalsteps,
        calories,
        total_minutes_asleep
    FROM sleepDay
    WHERE totalsteps IS NOT NULL
    AND calories IS NOT NULL
    ORDER BY totalsteps DESC
""")

,totalsteps,calories,total_minutes_asleep
0,36019,2690,NaN
1,29326,4547,NaN
2,27745,4398,NaN
3,23629,3808,NaN
4,23186,3921,NaN
...,...,...,...
858,17,257,302.0
859,16,1990,NaN
860,9,1843,NaN
861,8,1349,NaN


### DEVICE USAGE / ENGAGEMENT

In [ ]:
run_query("""
    SELECT
        id,
        COUNT(*) AS days_logged,
        CASE
            WHEN COUNT(*) >= 25 THEN 'High Engagement'
            WHEN COUNT(*) >= 15 THEN 'Moderate Engagement'
            ELSE 'Low Engagement'
        END AS engagement_level
    FROM sleepDay
    GROUP BY id
    ORDER BY days_logged DESC
""")

,id,days_logged,engagement_level
0,8877689391,31,High Engagement
1,8378563200,31,High Engagement
2,8053475328,31,High Engagement
3,6962181067,31,High Engagement
4,5553957443,31,High Engagement
5,4558609924,31,High Engagement
6,4445114986,31,High Engagement
7,4388161847,31,High Engagement
8,4319703577,31,High Engagement
9,2873212765,31,High Engagement


### WEIGHT LOG SUMMARY

In [ ]:
run_query("""
    SELECT
        ROUND(AVG(weightkg), 2)      AS avg_weight_kg,
        ROUND(AVG(bmi), 2)           AS avg_bmi,
        ROUND(MIN(bmi), 2)           AS min_bmi,
        ROUND(MAX(bmi), 2)           AS max_bmi,
        COUNT(DISTINCT id)           AS users_tracked_weight
    FROM weight_log
""")

,avg_weight_kg,avg_bmi,min_bmi,max_bmi,users_tracked_weight
0,72.04,25.19,21.45,47.54,8


### Results for Power BI

In [ ]:
activity_by_day = run_query("""
    SELECT day_of_week,
           ROUND(AVG(totalsteps), 0) AS avg_steps,
           ROUND(AVG(calories), 0)   AS avg_calories
    FROM sleepDay
    GROUP BY day_of_week
""")

activity_segmentation = run_query("""
    SELECT id,
           ROUND(AVG(totalsteps), 0) AS avg_steps,
           CASE
               WHEN AVG(totalsteps) < 5000  THEN 'Sedentary'
               WHEN AVG(totalsteps) < 7500  THEN 'Low Active'
               WHEN AVG(totalsteps) < 10000 THEN 'Fairly Active'
               ELSE 'Very Active'
           END AS activity_level
    FROM sleepDay
    GROUP BY id
""")

In [ ]:
# Export to Google Drive
output_dir = '/content/drive/MyDrive/Bellabeat'

activity_by_day.to_csv(f'{output_dir}/sql_activity_by_day.csv',          index=False)
activity_segmentation.to_csv(f'{output_dir}/sql_activity_segments.csv',  index=False)